# PART I: Data Processing    

In [20]:
import torch
import torchvision

In [21]:
# reproducibility
SEED = 0
# data path
PET_PATH = "../data/PetImages"
CAT_PATH = PET_PATH + "Cat"
DOG_PATH = PET_PATH + "Dog"

# train/val/test 0.7/0.2 the rest is for test
TRAIN_VAL_SPLIT = [0.7, 0.2]

# initial hyperparameters
BATCH_SIZE = 32
EPOCHS = 10
LEARNING_RATE = 1e-3

# scheduler hyperparameters
STRP_SIZE = 5
GAMMA = 0.1

In [22]:
# preprocessing transform to ImageNet-style normalization
from torchvision import transforms

train_trans = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229,0.224,0.225],
    ),
])

val_test_trans = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])


In [23]:
# define datasets, loaders
from torchvision.datasets import ImageFolder
from torch.utils.data import random_split
from torch.utils.data import DataLoader
# dataset
pet_dataset = ImageFolder(PET_PATH)
# create a split list
split_ls = [int(len(pet_dataset) * r) for r in TRAIN_VAL_SPLIT]
split_ls.append(len(pet_dataset)-sum(split_ls))
# split into subsets
train_set, val_set, test_set = random_split(pet_dataset, split_ls)
# specify transforms
train_set.dataset.transforms = train_trans
val_set.dataset.transforms = val_test_trans
test_set.dataset.transforms = val_test_trans
# loaders
train_loader = DataLoader(
    train_set,
    batch_size = BATCH_SIZE,
    shuffle = True,
    num_workers = 4,
)
val_loader = DataLoader(
    val_set,
    batch_size = BATCH_SIZE,
    shuffle = False,
    num_workers = 4
)
test_loader = DataLoader(
    test_set,
    batch_size = BATCH_SIZE,
    shuffle = False,
    num_workers = 4
)


In [24]:
# load pretrained model
from torchvision import models

weights = models.MobileNet_V3_Large_Weights.DEFAULT
model = models.mobilenet_v3_large(weights=weights)

print(model.classifier)

Sequential(
  (0): Linear(in_features=960, out_features=1280, bias=True)
  (1): Hardswish()
  (2): Dropout(p=0.2, inplace=True)
  (3): Linear(in_features=1280, out_features=1000, bias=True)
)


In [ ]:
# replace the last fully connected layer with our own
num_in_features = model.classifier[3].in_features

from torch import nn
fc_layer = nn.Linear(num_in_features, out_features=2, bias=True)
model.classifier[3] = fc_layer

# freeze backbone
for param in model.parameters():
    param.requires_grad = False

for param in model.classifier[3].parameters():
    param.requires_grad = True

# loss, optimizer
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    model.fc.parameters(),
    lr = LEARNING_RATE
)

scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=STEP_SIZE,
    gamma=GAMMA
)



AttributeError: 'MobileNetV3' object has no attribute 'fc'

In [ ]:
def train_one_epoch(model, loader):
    model.train()

